# Context Engineering

A language model knows two things when it answers: what it learned in training, and what is in its **context**, the text of the current call. Notebook 01 worked on the wording of the instructions, and notebook 02 added passages from a handbook. Both are parts of the context, and they are not the only ones: an example of the answer you want is one, the conversation so far is another, and all of it has to fit into a limited number of tokens.

**Context engineering** is the practice of deciding what goes into the context for each call: which instructions, which documents, how much of the conversation, in which form. In this notebook you build the context for a Tretwerk support assistant that remembers the conversation and looks up the handbook.

## Learning Objectives

By the end of this notebook, you should be able to:

- Compare the token cost of a whole document and of retrieved passages in the context.
- Teach a model an output format with an example in the context.
- Keep a conversation history and shorten it with a window or a summary.
- Rewrite follow-up questions so that retrieval understands them.

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()  # reads GROQ_API_KEY from the .env file

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=1000,
    max_retries=5,
    reasoning_effort="low",
)

The cell below builds the handbook retriever of notebook 02, section 6: one document per page, chunks of at most 400 characters, and a Chroma store that returns the three closest chunks.

In [ ]:
pages = sorted((Path("data") / "handbook").glob("*.md"))
documents = [
    Document(page_content=path.read_text(), metadata={"page": path.stem})
    for path in pages
]
chunks = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=0
).split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
store = Chroma.from_documents(chunks, embeddings, collection_name="handbook")
retriever = store.as_retriever(search_kwargs={"k": 3})


def format_passages(chunks):
    """Number the retrieved chunks."""
    return "\n\n".join(
        f"[{i}] {chunk.page_content}" for i, chunk in enumerate(chunks, start=1)
    )

## 1. What the Model Sees

Every call to the API is independent. The model receives one sequence of tokens, and everything it should take into account has to be in it:

<img src="assets/context_parts.png" alt="One call to the model contains, in this order: the system prompt with role and rules, which changes rarely; knowledge such as retrieved passages, which changes with every question; examples of an input and the answer you want, which change rarely; the conversation history, which grows with every turn; tool results, which change with every call; and the question." width="750">

The **context window** is the maximum number of tokens the model can read in one call; for `gpt-oss-20b` it is 131,072 tokens, about 100,000 words. That seems large enough for any task, but three costs grow with every token in the context: the price, the time until the answer, and the risk that the model overlooks the relevant part. The free Groq plan adds a fourth: 8,000 tokens per minute.

## 2. The Whole Handbook or a Few Passages?

The Tretwerk handbook is short, so it fits into the context as a whole. Is retrieval still worth it? The cell below asks the same question twice: once with the whole handbook in the system prompt, once with the three retrieved passages, and compares the input tokens.

In [ ]:
handbook = "\n\n".join(path.read_text() for path in pages)
question = "What does it cost to pause my subscription?"

whole = llm.invoke(
    [
        (
            "system",
            "You are a support agent at Tretwerk. Answer only from the handbook below.\n\n"
            + handbook,
        ),
        ("human", question),
    ]
)
retrieved = llm.invoke(
    [
        (
            "system",
            "You are a support agent at Tretwerk. Answer only from the passages below.\n\n"
            + format_passages(retriever.invoke(question)),
        ),
        ("human", question),
    ]
)

pd.DataFrame(
    [
        {
            "context": "whole handbook",
            "input tokens": whole.usage_metadata["input_tokens"],
            "answer": whole.content,
        },
        {
            "context": "3 passages",
            "input tokens": retrieved.usage_metadata["input_tokens"],
            "answer": retrieved.content,
        },
    ]
)

**Result.** Both answers are correct: 9 euros per month for storage. The whole handbook cost 2,670 input tokens, the three passages about 290, roughly nine times fewer, for the same answer. With the whole handbook, the free plan's 8,000 tokens per minute allow about three questions per minute; with passages, more than twenty. For a 20-page handbook, putting everything into the context is a reasonable choice, and it avoids the retrieval failures of notebook 02. For a knowledge base of thousands of pages it is impossible, and retrieval is the only option.

## 3. Examples in the Context

Section 1 listed what a call carries, and section 2 measured what it costs. One of the parts in that list is an **example**, and like everything else in the context it is paid for in tokens.

Describing a format in words works for simple formats. For an exact one an example is more reliable: the model continues the pattern it sees. This is **in-context learning**: the model learns the task from its context without training, **one-shot** prompting with one example, **few-shot** with several.

With a chat model an example is a pair of messages: a human message with an input and an ai message with the answer you want, as if the model had written it before. The ticket line below turns a customer email into a fixed format a support tool could read. The emails are the same customer emails as in notebook 01: `data/emails.csv` has a `text` column and a `category` column with the label each one should get.

In [ ]:
# the customer emails the ticket prompt turns into one line each
emails = pd.read_csv(Path("data") / "emails.csv")

In [ ]:
ticket_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You turn customer emails to Tretwerk into one ticket line."),
        (
            "human",
            "My battery only lasts 20 km since the cold weather started. Is it broken?",
        ),
        ("ai", "REPAIR | low | Battery range dropped to 20 km in cold weather"),
        ("human", "{email}"),
    ]
)

for i in [2, 4]:
    print(llm.invoke(ticket_prompt.invoke({"email": emails.loc[i, "text"]})).content)

**Result.** Both lines follow the example's format exactly: a category in capitals, a priority, a short summary, separated by `|`. The categories are wrong: the model invented `REPORT` for the stolen bike and filed the missed delivery under `REPAIR`. One example teaches the format but cannot show which labels exist; for that the prompt has to list them, as a classification prompt's instruction would.

## 4. The Model Has No Memory

A chat application appears to hold a conversation, but the API does not remember anything between calls. The two calls below are sent one after the other.

In [ ]:
system = (
    "system",
    "You are a support agent at Tretwerk, a company that rents e-bikes by subscription. Answer in at most two sentences.",
)

print(
    llm.invoke(
        [system, ("human", "Hi, I'm Sam. I have the Cargo plan in Cologne.")]
    ).content
)
print(llm.invoke([system, ("human", "Which plan do I have?")]).content)

**Result.** The first call answers Sam by name. The second call does not know who is asking: it asks for an email address to look up the plan. Each call contains only the messages sent with it, and the first message was not part of the second call.

To hold a conversation, the application keeps the **history** itself and sends it with every call: the user's messages and the model's answers, in order. This is all that "memory" means for a chat model. The function below appends each question and answer to a list and records how many input tokens each call used.

In [ ]:
def chat(history, question):
    """Send the system prompt, the history and the question; append both messages to the history."""
    reply = llm.invoke([system, *history, ("human", question)])
    history.extend([("human", question), ("ai", reply.content)])
    return reply.content, reply.usage_metadata["input_tokens"]


conversation = [
    "Hi, I'm Sam. I have the Cargo plan in Cologne.",
    "My daughter is three. Can she ride in the box?",
    "And can I take the bike into the train?",
    "What was my name again, and which plan do I have?",
]

history = []
for question in conversation:
    reply, tokens = chat(history, question)
    print(f"[{tokens} input tokens] {question}\n  -> {reply}\n")

**Result.** With the history, the fourth answer knows Sam's name and plan. The input tokens show the cost: the same short system prompt grows from 114 tokens in the first turn to 310 in the fourth, because every call repeats the whole conversation. Note also what the model gets wrong: there is no handbook in this context, so it invents the rules. It says that a three-year-old should not ride in the cargo box, while Tretwerk's handbook says that the box has seat belts for two children. Section 6 adds the handbook to the context again.

## 5. Keep the History Short

The full history grows with every turn, and a long support conversation would soon cost more tokens for the history than for anything else. Two common ways to keep it short:

- a **window** keeps only the last few messages and drops the rest;
- a **summary** replaces the older messages with a short summary written by the model.

<img src="assets/memory_strategies.png" alt="With the fourth question, the full history sends all six earlier messages. The window sends only the last question and answer, so the first message with Sam's name and plan is gone. The summary sends a summary plus the last question and answer." width="1000">

A window is simple: it slices the list. The next cell keeps the last two messages, one question and one answer, and runs the same conversation.

In [ ]:
def chat_window(history, question, keep=2):
    """Like chat, but send only the last `keep` messages of the history."""
    reply = llm.invoke([system, *history[-keep:], ("human", question)])
    history.extend([("human", question), ("ai", reply.content)])
    return reply.content, reply.usage_metadata["input_tokens"]


history = []
for question in conversation:
    reply, tokens = chat_window(history, question)
    print(f"[{tokens} input tokens] {question}\n  -> {reply}\n")

**Result.** The window keeps the input tokens almost flat: 183 and 186 in the last two turns, against 227 and 310 with the full history. The disadvantage is visible in the last answer: Sam's name and plan were in the first message, which the window dropped, so the model asks for them again. A window works for conversations in which only the recent turns matter.

**Your turn:** build summary memory. Instead of dropping the older messages, `chat_summary` keeps a running `summary` of the conversation and sends it in the system prompt, together with only the last two messages. After each answer, a second call updates the summary. Complete the prompt and the two calls, then run the conversation.

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "Update the summary of a support conversation. Keep every fact about the customer, such as their name and plan. "
                "Use at most three sentences."
            ),
        ),
        # TODO: a human message with the placeholders {summary}, {question} and {reply}
        ...,
    ]
)


def chat_summary(state, question):
    """Answer with the summary and the last two messages, then update the summary."""
    context = (
        "system",
        system[1] + "\n\nSummary of the conversation so far: " + state["summary"],
    )
    reply = ...  # TODO: call the model with the context, the last two messages of state["history"] and the question
    state["history"].extend([("human", question), ("ai", reply.content)])
    state[
        "summary"
    ] = ...  # TODO: call summary_prompt | llm | StrOutputParser() with the old summary, the question and the reply
    return reply.content, reply.usage_metadata["input_tokens"]


state = {"summary": "(none yet)", "history": []}
for question in conversation:
    reply, tokens = chat_summary(state, question)
    print(f"[{tokens} input tokens] {question}\n  -> {reply}\n")
print("SUMMARY:", state["summary"])

## 6. Follow-Up Questions in RAG

Follow-up questions are where the history and retrieval interact. In a conversation, people write *"How much does that cost?"* and mean the thing they just talked about. The model can resolve *that* from the history, but the retriever cannot: it embeds only the question it gets.

In [ ]:
history = [
    ("human", "Can I pause my subscription over the winter?"),
    ("ai", "Yes, you can pause it for up to two months per calendar year."),
]
follow_up = "How much does that cost?"

for chunk in retriever.invoke(follow_up):
    print(chunk.metadata["page"], "|", chunk.page_content[:80].replace("\n", " "))

**Result.** The retriever returns the subscription prices, the accessories and the theft page, none of which is about pausing. It embedded *How much does that cost?* on its own, and the closest chunks are those that mention costs.

The solution is one more call before the retrieval: the model **rewrites** the follow-up into a question that can be understood on its own, using the history. The `("placeholder", "{history}")` entry in the template inserts the list of history messages at that position.

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the user's last question so that it can be understood without the conversation. Return only the rewritten question.",
        ),
        ("placeholder", "{history}"),
        ("human", "{question}"),
    ]
)
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

standalone = rewrite_chain.invoke({"history": history, "question": follow_up})
print(standalone)
for chunk in retriever.invoke(standalone):
    print(chunk.metadata["page"], "|", chunk.page_content[:80].replace("\n", " "))

**Result.** The model turns the follow-up into *What is the cost of pausing my subscription?*, and the retriever now returns both chunks of the pausing page, including the one with the fee. Rewriting costs one extra call per question, which is why it is only needed once the conversation has a history.

**Your turn:** combine these parts into a support assistant. `support_chat(history, question)` should rewrite the question with the history, retrieve passages for the rewritten question, answer from the passages with the history in the context, and append the original question and the answer to the history. Run it on the three-turn conversation below.

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "You are a support agent at Tretwerk. Answer only from the numbered passages and cite them, for example [1]. "
                "If they do not contain the answer, say that the handbook does not cover it.\n\nPassages:\n{passages}"
            ),
        ),
        ("placeholder", "{history}"),
        ("human", "{question}"),
    ]
)


def support_chat(history, question):
    # TODO: rewrite, retrieve, answer, update the history, and return the answer
    ...


history = []
for question in [
    "Can I pause my subscription over the winter?",
    "How much does that cost?",
    "And who stores the bike in the meantime?",
]:
    print(f"{question}\n  -> {support_chat(history, question)}\n")

## Summary

- The **context** is everything the model receives in one call: system prompt, knowledge, examples, conversation history, tool results and the question. **Context engineering** decides what goes in, for every call.
- A whole document can fit into the **context window**, but every token is paid for in every call. Here the whole handbook cost 2,670 input tokens for a question that three retrieved passages answered with about 290.
- An **example** in the context teaches a format more reliably than describing it, and costs the tokens it occupies. One example cannot show which labels exist.
- The API is **stateless**: a conversation exists only because the application sends the history with every call, so the cost of a conversation grows with every turn.
- A **window** keeps the last messages and forgets the rest, including facts the user gave at the start. A **summary** keeps the facts in a few sentences, at the cost of one more call per turn.
- **Follow-up questions** fail in retrieval because the retriever sees only the question. Rewriting the question with the history first makes it retrievable.

## Sources

- Liu, N. F. et al. (2023). [Lost in the Middle: How Language Models Use Long Contexts](https://arxiv.org/abs/2307.03172): models use information at the start and the end of a long context better than in its middle.
- Anthropic (2025). [Effective Context Engineering for AI Agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents).
- [Groq documentation for `gpt-oss-20b`](https://console.groq.com/docs/model/openai/gpt-oss-20b), with its context window and limits.
- Tretwerk, its handbook and its emails are fictional, written for this repository.